Treinamento do autoencoder genérico sem mascaramento de derivações

---

Bibliotecas

In [85]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [86]:
import torch
import torch.nn as nn

from torch.utils.data import random_split
from torch.utils.data import DataLoader

from sklearn.metrics  import r2_score

from src.Dataset import Code15Dataset
from src.utils   import EarlyStopping

---

Preparação do dataset

In [87]:
DATA_FOLDER = "../data"
BATCH_SIZE  = 32

In [88]:
train_test_files = os.listdir(DATA_FOLDER)
train_test_files = train_test_files[:-1]

In [89]:
train_test_dataset = Code15Dataset(train_test_files, DATA_FOLDER)

Hold-out

In [90]:
train_size = int(0.8 * len(train_test_dataset))
test_size  = len(train_test_dataset) - train_size

train_set, val_set = random_split(
    train_test_dataset,
    [train_size, test_size]
)

Dataloaders

In [91]:
train_loader = DataLoader(
    train_set,
    batch_size = BATCH_SIZE,
    shuffle    = True
)

val_loader = DataLoader(
    val_set,
    batch_size = BATCH_SIZE,
    shuffle    = False
)

---

Autoencoder genérico

In [92]:
class GenericAutoencoder(nn.Module):

    def __init__(self, latentDim):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv1d(12, 32, 5, stride=2, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, 5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, 5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, latentDim, 5, stride=2, padding=2),
            nn.BatchNorm1d(latentDim),
            nn.ReLU()

        )
        
        self.decoder = nn.Sequential(

            nn.ConvTranspose1d(latentDim, 128, 4, stride=2, padding=1),
            nn.ReLU(),

            nn.ConvTranspose1d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),

            nn.ConvTranspose1d(32, 12, 4, stride=2, padding=1),

        )

    def forward(self, x):
        x  = x.permute(0, 2, 1)
        z   = self.encoder(x)
        out = self.decoder(z)
        
        return out.permute(0, 2, 1)

In [93]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [94]:
model = GenericAutoencoder(
    latentDim = 128
)

model = torch.compile(model)

model = model.to(device)


In [95]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

---

Treinamento

In [96]:
def compute_r2(y_true, y_pred):

    y_true = y_true.detach().cpu().flatten(0, 1).numpy()
    y_pred = y_pred.detach().cpu().flatten(0, 1).numpy()

    return r2_score(y_true, y_pred)

In [97]:
def save_best_checkpoint(
    model,
    optimizer,
    epoch,
    val_r2,
    dist_dir
):
    os.makedirs(dist_dir, exist_ok=True)

    checkpoint_path = os.path.join(
        dist_dir,
        "best_model.pth"
    )

    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_r2":               val_r2
    }, checkpoint_path)

In [98]:
def train_epoch(model, dataloader, optimizer, criterion, device):

    model.train()

    epoch_loss = 0
    epoch_r2   = 0

    for X in dataloader:
        X = X.to(device)

        prediction = model(X)

        loss = criterion(prediction, X)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_r2   += compute_r2(X, prediction)

    epoch_loss /= len(dataloader)
    epoch_r2   /= len(dataloader)

    return epoch_loss, epoch_r2


In [99]:
def evaluate(model, dataloader, criterion, device):

    model.eval()

    loss = 0
    r2   = 0

    with torch.no_grad():
        for X in dataloader:

            X = X.to(device)

            prediction = model(X)

            batch_loss = criterion(prediction, X)

            loss += batch_loss.item()
            r2   += compute_r2(X, prediction)

    loss /= len(dataloader)
    r2   /= len(dataloader)

    return loss, r2

In [100]:
def train(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs,
    dist_dir
):

    losses    = []
    r2_scores = []

    best_val_r2 = -float("inf")

    early_stopping = EarlyStopping(
        patience  = 10,
        min_delta = 1e-3
    )

    for epoch in range(epochs):

        train_loss, train_r2 = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_loss, val_r2 = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        losses.append(train_loss)
        r2_scores.append(train_r2)

        print(
            f"Epoch={epoch} "
            f"train_loss={train_loss:.5f} "
            f"train_r2={train_r2:.5f} "
            f"val_loss={val_loss:.5f} "
            f"val_r2={val_r2:.5f}"
        )

        if val_r2 > best_val_r2:

            best_val_r2 = val_r2

            save_best_checkpoint(
                model,
                optimizer,
                epoch,
                val_r2,
                dist_dir
            )

            print(
                f"New best model saved "
                f"(epoch={epoch}, val_r2={val_r2:.5f})"
            )

        early_stopping(val_r2)

        if early_stopping.stop:

            print(
                f"Early stopping triggered at epoch {epoch}"
            )

            break

    return losses, r2_scores

In [101]:
losses, r2_scores = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    100,
    "../dist"
)

KeyboardInterrupt: 